<a href="https://colab.research.google.com/github/erinijapranckeviciene/idkwdan/blob/main/code/Countries_keywords_classification_with_embedding.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Word embedding learning from article keywords.
Data generated in publication DOI: 10.3390/ijerph21091202

### Country labels

In [ ]:
#Assign class names
class_names=['Afghanistan','Albania','Algeria','Argentina','Armenia','Australia','Austria','Azerbaijan',
             'Bahamas','Bahrain','Bangladesh','Barbados','Belarus','Belgium','Benin','Bhutan','Bolivia',
             'Bosnia_and_Herzegovina','Brazil','Brunei_Darussalam','Bulgaria','Burkina_Faso','Cambodia',
             'Cameroon','Canada','Central_African_Republic','Chad','Chile','China','Colombia','Congo',
             'Congo_Democratic_Republic','Costa_Rica','Croatia','Cuba','Cyprus','Czechia','Denmark',
             'Dominican_Republic','Ecuador','Egypt','El_Salvador','Eritrea','Estonia','Ethiopia','Fiji',
             'Finland','France','Gambia','Georgia','Germany','Ghana','Greece','Guatemala','Guinea','Guinea-Bissau',
             'Guyana','Haiti','Honduras','Hungary','Iceland','India','Indonesia','Iran','Iraq','Ireland','Israel',
             'Italy','Jamaica','Japan','Jordan','Kazakhstan','Kenya','Kiribati','Kuwait','Kyrgyz_Republic',
             'Latvia','Lebanon','Lesotho','Liberia','Libya','Lithuania','Luxembourg','Madagascar','Malawi',
             'Malaysia','Mali','Malta','Mauritania','Mexico','Micronesia','Moldova','Mongolia','Montenegro',
             'Morocco','Mozambique','Myanmar','Namibia','Nepal','Netherlands','Nicaragua','Niger','Nigeria',
             'North_Korea','Norway','Oman','Pakistan','Panama','Papua_New_Guinea','Paraguay','Peru',
             'Philippines','Poland','Portugal','Qatar','Romania','Russian_Federation','Rwanda','Samoa',
             'Saudi_Arabia','Senegal','Serbia','Seychelles','Sierra_Leone','Singapore','Slovak_Republic',
             'Slovenia','Solomon_Islands','Somalia','South_Africa','South_Korea','Spain','Sri_Lanka',
             'St._Lucia','St_Vincent_Grenadines','Sudan','Suriname','Sweden','Switzerland','Syria',
             'Tajikistan','Tanzania','Thailand','Togo','Tonga','Trinidad_and_Tobago','Turkey','Uganda',
             'Ukraine','United_Arab_Emirates','United_Kingdom','United_States','Uruguay','Uzbekistan',
             'Vanuatu','Venezuela','Vietnam','Yemen','Zimbabwe']


In [ ]:
import os, pathlib, shutil, random
import tensorflow as tf
from tensorflow import keras

batch_size = 32
train_ds = keras.utils.text_dataset_from_directory("absolutepath/to/kwd_per_countries_training_data/countries/", batch_size=batch_size, label_mode="categorical", shuffle=True, labels="inferred", class_names=class_names)

#val_ds = keras.utils.text_dataset_from_directory("kwdsuic/train", batch_size=batch_size, label_mode="categorical")
#test_ds = keras.utils.text_dataset_from_directory("kwdsuic/train", batch_size=batch_size, label_mode="categorical")

#### Inspect the dataset read from directory files
Displaying the shapes and dtypes of the first batch

In [ ]:
for inputs, targets in train_ds:
    print("inputs.shape:", inputs.shape)
    print("inputs.dtype:", inputs.dtype)
    print("targets.shape:", targets.shape)
    print("targets.dtype:", targets.dtype)
    print("inputs[0]:", inputs[0])
    print("targets[0]:", targets[0])
    break

#### Keep only text in text_only_train_ds. Use only text data.

In [ ]:
# This function returns only data part without target
# to create a new dataset that will be used to create dictionary
text_only_train_ds = train_ds.map(lambda x, y: x)
for inputs in text_only_train_ds:
    print("inputs.shape:", inputs.shape)
    print("inputs.dtype:", inputs.dtype)
    print("inputs[0]:", inputs[0])
    break


## Keyword sequences must be transformmed into sequences of indices of the vocabulary words to use embedding
To use embedding we need fixed sequence length. Each country must have country_article_no.txt. For this dataset was transformed.

##### Later try to classify countries .

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers

# max length of keywords sequence
max_length = 50
# max length of the vocabulary
max_tokens = 60000

# create text vectorization layer in which words in training data set are encoded as indices in the vocabulary
text_vectorization = layers.TextVectorization( max_tokens=max_tokens, output_mode="int",
                                              output_sequence_length=max_length, standardize=None)

# Create vocabulary
text_vectorization.adapt(text_only_train_ds)

# Create data sets - in our case all datasets are the same
int_train_ds = train_ds.map(lambda x, y: (text_vectorization(x), y), num_parallel_calls=4)
int_val_ds = train_ds.map(lambda x, y: (text_vectorization(x), y), num_parallel_calls=4)
int_test_ds = train_ds.map(lambda x, y: (text_vectorization(x), y), num_parallel_calls=4)

#### This code shows how inputs and targets look like

In [ ]:
for inputs, targets in int_train_ds:
    print(inputs.shape)
    print(inputs[0])
    print(targets[0])
    # With this test_input variable verify tf.one_hot() transformation
    test_input=inputs[0]
    break

#### This is to show how to retrieve a vocabulary

In [ ]:
vc=text_vectorization.get_vocabulary()
for tg in vc:
    if 'gene' in tg:
        print(tg)

print(len(vc))

#### Use Enbedding layer to encode words , to represent each word by a semantic gradient
The keywords will be treated as sequences

In [ ]:
vocab_size = len(vc)
embed_dim = 300
hidden_dim = 100
max_length = 50


# ignore from FC book Ch.11 Listing 11.22
#num_heads = 2
#dense_dim = 32

inputs = keras.Input(shape=(max_length), dtype="int64")

embedded = layers.Embedding(vocab_size, embed_dim, input_length=max_length, name="embedded" )(inputs)

x = layers.Bidirectional(layers.LSTM(200))(embedded)
#x=layers.LSTM(32)(embedded)
#x=layers.Flatten()(embedded)
#x = layers.Dense(hidden_dim, activation="relu")(x)
#x = layers.Dense(2*hidden_dim, activation="relu")(x)

x = layers.Dropout(0.1)(x)
outputs = layers.Dense(159, activation="softmax")(x)

model = keras.Model(inputs, outputs)

# Maybe optimizer could be different?
model.compile(optimizer="rmsprop", loss="categorical_crossentropy", metrics=["accuracy"])
model.summary()

In [ ]:
callbacks = [ keras.callbacks.ModelCheckpoint("int_embedded_lstm_countries_keywords_only.keras", save_best_only=True) ]
model.fit(int_train_ds, validation_data=int_train_ds, epochs=20, verbose=1, callbacks=callbacks)

In [ ]:
model = keras.models.load_model("int_embedded_lstm_countries_keywords_only.keras")
print(f"Test acc: {model.evaluate(int_train_ds)[1]:.3f}")

#### Save the embedding vectors with vocabulary

In [ ]:
import pandas as pd
# extract the embedded layer
l=model.get_layer("embedded").output
print(l)
# extract the weights
# each weight vector corresponds to a word of the vocabulary
w=model.get_layer("embedded").get_weights()

#### Write files for Embedded Projector

In [ ]:
weights = model.get_layer('embedded').get_weights()[0]
vocab = text_vectorization.get_vocabulary()

In [ ]:
import io

# create vocabulary
embedded_vectors={}

out_v = io.open('vectors.tsv', 'w', encoding='utf-8')
out_m = io.open('metadata.tsv', 'w', encoding='utf-8')

for index, word in enumerate(vocab):
  if index == 0:
    continue  # skip 0, it's padding.
  vec = weights[index]
  out_v.write('\t'.join([str(x) for x in vec]) + "\n")
  out_m.write(word + "\n")
  embedded_vectors[word]=vec.tolist()

out_v.close()
out_m.close()


In [ ]:
#print(embedded_vectors['validation'])

#### Save embedding vectors dictionary

In [ ]:
import json
file='embedded_vectors.json'

with open(file, 'w') as f:
    json.dump(embedded_vectors, f)

# How to read it from json file
#with open(file, 'r') as f:
#    data = json.load(f)

## Following is extra analysis
#### Get the confusion table

#### Recreate the data set without shuffling
This is needed because the dataset batches must not be shuffled in order the targets of samples to correspond to the predicted labels of the same samples.  

In [ ]:
import os, pathlib, shutil, random
import tensorflow as tf
from tensorflow import keras

batch_size = 32
train_ds_for_testing = keras.utils.text_dataset_from_directory("../kwdsuic/kwd_per_countries/train", batch_size=batch_size, label_mode="categorical", shuffle=False,
                                                  labels="inferred", class_names=class_names )

#val_ds = keras.utils.text_dataset_from_directory("kwdsuic/train", batch_size=batch_size, label_mode="categorical")
#test_ds = keras.utils.text_dataset_from_directory("kwdsuic/train", batch_size=batch_size, label_mode="categorical")

#### Transform textual data set into

In [ ]:
int_train_ds_for_testing = train_ds_for_testing.map(lambda x, y: (text_vectorization(x), y), num_parallel_calls=4)

In [ ]:
import numpy as np
pred=model.predict(int_train_ds_for_testing)
predicted = pred.argmax(axis=-1)

print(np.array(class_names)[predicted])


In [ ]:
import numpy as np
# labels from dataset
actual = np.argmax(np.concatenate([y for x, y in int_train_ds_for_testing], axis=0), axis=1)
print(len(actual) )


In [ ]:
inp=np.concatenate([x for x, y in int_train_ds_for_testing])

In [ ]:
print(inp)

In [ ]:
kwds=[]
ind=[]
for samp in inp:
    i =[x for x in samp if x>0]
    w =[vc[x] for x in samp if x>0]
    id=' '.join(str(i) )
    ws=' '.join(w)
    kwds.append(ws)
    ind.append(id)


In [ ]:
import pandas as pd
df=pd.DataFrame({'actual': actual,'predicted':predicted,
                 'actual_country': np.array(class_names)[actual],
                 'predicted_country': np.array(class_names)[predicted],
                 'kwds':kwds, 'ind':ind})

In [ ]:
sameclass=np.multiply(actual==predicted,1)
print(sameclass)

In [ ]:
df['sameclass']=sameclass

In [ ]:
print(df.shape[0])

In [ ]:
df.head

#### Make country summaries for each word in vocabulary

In [ ]:
words_by_country={}
actual_country=df['actual_country']
keywords=df['kwds']

for index, kwds in enumerate(keywords):
    country=actual_country[index]
    for kwd in kwds.split(' '):
        if kwd in words_by_country.keys():
            # verify if country exists
            if country in words_by_country[kwd].keys():
                words_by_country[kwd][country]+=1
            else:
                words_by_country[kwd][country]=1
        else:
            words_by_country[kwd]={}
            words_by_country[kwd][country]=1

In [ ]:
print(words_by_country)

In [ ]:
# Make data frame
dfc=pd.DataFrame(words_by_country.values() )
dfc=dfc.fillna(0)

n_countries=[]
word_freq=[]

for i in dfc.index:
    nc=sum(dfc.iloc[i]>0 )
    wf=sum(dfc.iloc[i] )
    n_countries.append(nc)
    word_freq.append(wf)

dfc['n_countries']=n_countries
dfc['word_freq']=word_freq
dfc['words']=words_by_country.keys()

dfc=dfc.sort_values(by='word_freq', ascending=False)
print(dfc)

dfc.to_csv('words_and_countries.csv', index=False)

In [ ]:
l=[x for x in n_countries if x<0]
print(l)

#### Try to filter most frequent words from keyword strings

In [ ]:
# Group and count actual and predicted
cid=[]
for j in range(df.shape[0]):
    element='_'.join([ str(df['actual'][j]), str(df['predicted'][j]) ] )
    cid.append(element)

df['cid']=cid

In [ ]:
df_g=df.copy()
# group the counts

df_groups=df_g.groupby("cid").agg(
    actual=pd.NamedAgg(column="actual", aggfunc="max"),
    actual_country=pd.NamedAgg(column="actual_country", aggfunc=lambda x: str(pd.unique(x)[0]) ),
    predicted=pd.NamedAgg(column="predicted", aggfunc="max"),
    predicted_country=pd.NamedAgg(column="predicted_country", aggfunc=lambda x: str(pd.unique(x)[0]) ),
    counts=pd.NamedAgg(column="cid", aggfunc="count"),
    sameclass=pd.NamedAgg(column="sameclass", aggfunc="max"),
    kwds=pd.NamedAgg(column="kwds",aggfunc=lambda x: ' '.join(x))
).sort_values(by=['counts'], ascending=False)

print(df_groups)
df_groups.to_csv('countries_kwd_groups.csv')

In [ ]:
cnt=df_groups['counts']
values, counts = np.unique(cnt, return_counts=True)
dfcnt=pd.DataFrame({'values':values, 'counts':counts}).sort_values(by=['counts'], ascending=False)
dfcnt

In [ ]:
# show confusion table
# Why confusion table shows different accuracy than accuracy of model evaluate?
# Need to investigate, but leaving it for later now.
cm=tf.math.confusion_matrix(actual, predicted).numpy()
print(cm)
cmdf=pd.DataFrame(cm, columns=class_names)
cmdf1=cmdf.copy()
cmdf['countries']=class_names
cmdf.set_index('countries')
cmdf.to_csv("countries_conf_mat.csv", index=False)

acc=sum(np.diagonal(cm))/np.sum(cm)
print(acc)

In [ ]:
# make normalized
cmdf_perc=cmdf1.div(cmdf1.sum(axis=1), axis=0)
#cmdf_perc['countries']=class_names
#cmdf_perc.set_index('countries')
#cmdf_perc.head

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

plt.style.use('ggplot')

np.random.seed(37)

#n = 10
#labels = [f'x{i}' for i in range(n)]
#df = pd.DataFrame(np.random.randint(0, 11, size=(n, n)), index=labels, columns=labels)

fig, ax = plt.subplots(figsize=(8, 8))

_ = sns.heatmap(cmdf_perc)
_ = ax.set_title('Confusion table heatmap plot')

y_min, y_max = ax.get_ylim()
dh = 0.8
_ = ax.set_ylim(y_min + dh, y_max - dh)

In [ ]:
cmdf_perc['countries']=class_names
cmdf_perc.set_index('countries')
cmdf_perc.to_csv("countries_conf_mat_perc.csv", index=True)
cmdf_perc.head

## Explore the embedding layer
The embedding layer dimensions correspond to the words in the vocabulary. The embedding vectors encode words by their semantic context and must have a nice structure where semantically similar words are represented by proximal vectors. Therefore, an expectation is that when we "umap" the embedding vectors, we will see clusters that contain semantically similar words in this specific context.   

In [ ]:
import pandas as pd
# how to get the layer
l=model.get_layer("embedded").output
print(l)
w=model.get_layer("embedded").get_weights()
for i in range(23787,23789):
    print(i)
    print(len(w[0][i] ) )
wn=w[0]

print(wn.shape )

#wdf=pd.DataFrame(w)
#vc=text_vectorization.get_vocabulary()
#for tg in vc:
#    print(tg)

print(len(vc))

#### UMAP

In [ ]:
import numpy as np
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import umap

reducer=umap.UMAP( n_neighbors=50, repulsion_strength=12)
#repulsion_strength=10
#scaled_data = StandardScaler().fit_transform(wn)

embedding = reducer.fit_transform(wn)

#embedding = reducer.fit_transform(scaled_data)
print(embedding.shape)



In [ ]:
plt.scatter(
    embedding[:, 0],
    embedding[:, 1])
#    c=actual)

#c=[sns.color_palette()[x] for x in penguins.species.map({"Adelie":0, "Chinstrap":1, "Gentoo":2})]

plt.gca().set_aspect('equal', 'datalim')
plt.title('UMAP projection of word embedding', fontsize=24);


#### t-SNE

In [ ]:
#help(TSNE)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
from sklearn.manifold import TSNE

tsne = TSNE(n_components=2, learning_rate='auto', init='pca', perplexity=200, early_exaggeration=300).fit_transform(wn)

#embedding = reducer.fit_transform(scaled_data)
print(tsne.shape)


In [ ]:
plt.scatter(
    tsne[:, 0],
    tsne[:, 1])
#    c=actual)

#c=[sns.color_palette()[x] for x in penguins.species.map({"Adelie":0, "Chinstrap":1, "Gentoo":2})]

plt.gca().set_aspect('equal', 'datalim')
plt.title('t-SNE projection of word embedding', fontsize=24);


#### What classes the vocabulary words belong to

In [ ]:
for i in range(1,3):
    a=[i]+[0]*29
    b=tf.constant(a)
    print(b)
    #c=model.predict(b)
    #print(c)

In [ ]:
import numpy as np

for inputs, targets in int_train_ds:
    print(inputs.shape)
    print(inputs[0])
    print(targets[0])
    # With this test_input variable verify tf.one_hot() transformation
    break
p=model.predict(inputs)
print(p[0])
print(targets[0])

for i in range(100):
    ind=[x for x in inputs[i].numpy() if x >0]
    words=[vc[i] for i in ind]
    print(ind)
    print(words)
    print(np.argmax(p[i]) , np.argmax(targets[i]) )
#target= np.argmax(targets, axis=1)